# ICU Mortality Prediction Model — **Phase 3 (baseline)**
### Regularized logistic regression on both variable sets, measured against the SAPS3 yardstick

For this admission, what is the risk that the
patient will not survive the hospitalization? This phase does not yet seek the most powerful model — it
seeks the honest, transparent baseline, of the same family as SAPS3, against which everything else will
be measured.*

We continue from where Phase 2 stopped: the base `modelo_A_fase2.csv` (**5,676** index admissions —
5,127 unique pairs plus 551 recovered by positional alignment in Phase 1) and the two variable lists
(`modelo_A_conjuntos_variaveis.json`, already including `OrigemPagamento`). This phase delivers, with
care and without haste:

1. **Input sanitization** — a check that detects and corrects Excel artifacts (the thousands separator),
   isolated in its own cell, easy to remove when the base comes clean.
2. **Main-diagnosis grouping** — the ~503 categories reduced to the 20 most frequent + "other",
   the same logic already used for comorbidities and secondary diagnoses.
3. **Encoding, imputation, and tail handling** — one-hot with a reference category, median imputation,
   and a cap on the urea/creatinine ratio before logistic regression.
4. **Two validations** — temporal as the primary verdict (train on the past, test on the future) and
   random by patient as a stability reference.
5. **Regularized logistic regression on both sets**, with SAPS3 as the reserved benchmark.

Gradient boosting, SHAP, the DeLong test, and fine calibration are left for the next phase.

**Note on the outcome:** the results of this phase use the primary definition of `mortalidade_hospitalar`
(`HospitalDischargeName`). Cardiovascular patients transferred before death may be coded as survivors
under this definition, underestimating true mortality in that subgroup — a transfer bias documented in
the Transfer Bias Note. The parallel analysis with the adjusted definition
(`mortalidade_hospitalar_ajustada`) will follow the same protocol as this phase and will be compared in
Phase 5. All results below refer to the primary definition unless explicitly stated otherwise.

In [ ]:
import os, json, warnings
import pandas as pd, numpy as np
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

# === Configuration ===
BASE_DIR   = 'outputs/phase2'
CSV_PATH   = f'{BASE_DIR}/modelo_A_fase2.csv'
CONJ_PATH  = f'{BASE_DIR}/modelo_A_sets_variaveis.json'
OUTDIR     = 'outputs/phase3'
os.makedirs(OUTDIR, exist_ok=True)

ALVO = 'mortalidade_hospitalar'
CORTE_TEMPORAL = 2023   # training < 2023; test >= 2023

## 1 · Input sanitization

This cell fixes a specific problem: Excel's thousands separator in numeric columns with long decimal
places, where values like 95.33 were rewritten as `95.333.333`. It affects two columns:

- **PaO₂/FiO₂** is recalculated from the source columns (`Highest Pa O21h` and `Highest Fi O21h`),
  which remained intact. In uncorrupted rows, the recalculation matches the original value with zero
  difference, confirming a faithful reconstruction.
- **Mean arterial pressure**, which is not a derived ratio, is reconstructed directly from the
  thousands format.

If the base is regenerated without passing through Excel, this cell will find nothing to fix and can be
removed with no effect.

In [ ]:
def sanear_residuos_excel(df, verbose=True):
    """Detecta e corrige o separador de milhar do Excel em colunas numéricas.
    Idempotente: se a base já estiver limpa, no altera nada."""
    df = df.copy()
    def tem_milhar(serie):
        return serie.astype(str).str.count(r'\.').ge(2).sum()

    # --- PaO2/FiO2: recalculate from the source (safer than undoing the thousands separator) ---
    col_ratio = 'Highest Pa O2Fi O21h'
    if col_ratio in df.columns and tem_milhar(df[col_ratio]) > 0:
        n = tem_milhar(df[col_ratio])
        pao2 = pd.to_numeric(df['Highest Pa O21h'], errors='coerce')
        fio2 = pd.to_numeric(df['Highest Fi O21h'], errors='coerce')  # FiO2 em %, 21-100
        df[col_ratio] = pao2 / (fio2/100)
        if verbose: print(f'  {col_ratio}: {n} valores recalculados a partir de PaO2 e FiO2 de origem')

    # --- MAP and any other columns with thousands separators: reconstruct the decimal ---
    def fix_milhar(v):
        if pd.isna(v): return np.nan
        v = str(v).strip()
        if v == '' or v.lower() == 'nan': return np.nan
        if v.count('.') >= 2:
            p = v.split('.')
            return float(f"{p[0]}.{''.join(p[1:])}")
        try: return float(v)
        except: return np.nan
    for col in df.columns:
        if col == col_ratio: continue
        if df[col].dtype == object and tem_milhar(df[col]) > 0:
            n = tem_milhar(df[col])
            df[col] = df[col].map(fix_milhar)
            if verbose: print(f'  {col}: {n} values reconstructed from the thousands format')
    return df

# Read everything as text first, so sanitization acts before any numeric conversion
raw = pd.read_csv(CSV_PATH, encoding='utf-8-sig', dtype=str)
print(f'Base carregada: {raw.shape[0]} linhas x {raw.shape[1]} colunas')
print('\nSaneamento de entrada:')
df = sanear_residuos_excel(raw)
print('  (if nothing was listed above, the base was already clean)')

### Type conversion and sanity of the sanitized columns
With the artifacts corrected, we convert everything that is not text to numeric type, and verify that
the two treated columns rest within clinically plausible ranges.

In [ ]:
COLUNAS_TEXTO = ['Gender','AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
    'AdmissionMainDiagnosisName','Chronic Health Status Name','Status COVID 19','Month Year',
    'EpimedCode','MedicalRecord','adm_datetime','HospitalDischargeName','UnitDischargeName',
    'Anatomic Tumor Site Name','HematologicalMalignancyTypeName','patient_id',
    'OrigemPagamento']   # nova variable da Fase 1 — precisa ficar de fora da conversão numérica

# Boolean flags from Phase 1/2 (turned into 'True'/'False' text in the CSV round-trip) — converted separately,
# never by the pd.to_numeric below (which would silently turn them into NaN).
COLUNAS_BOOL = ['par_unico','integravel','recuperado_por_posicao','folha8_integravel']
for c in COLUNAS_BOOL:
    if c in df.columns:
        df[c] = df[c].map({'True': True, 'False': False})

for c in df.columns:
    if c not in COLUNAS_TEXTO and c not in COLUNAS_BOOL:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['adm_datetime'] = pd.to_datetime(df['adm_datetime'], errors='coerce')
df['ano'] = df['adm_datetime'].dt.year

print('Sanity of the sanitized columns (expected clinical ranges):')
print(f"  PAM (mmHg):   mediana={df['Lowest Mean Arterial Pressure1h'].median():.1f}  "
      f"[min={df['Lowest Mean Arterial Pressure1h'].min():.1f}, max={df['Lowest Mean Arterial Pressure1h'].max():.1f}]")
print(f"  PaO2/FiO2:    mediana={df['Highest Pa O2Fi O21h'].median():.1f}  "
      f"[min={df['Highest Pa O2Fi O21h'].min():.1f}, max={df['Highest Pa O2Fi O21h'].max():.1f}]")
print(f"\nDesfecho — deaths: {int(df[ALVO].sum())} ({df[ALVO].mean()*100:.1f}%)  |  nulos: {df[ALVO].isna().sum()}")

## 2 · Main-diagnosis grouping (top-20 + other)

`AdmissionMainDiagnosisName` carries 503 distinct diagnoses, but the distribution is a long tail: 204
appear in a single patient, and the 20 most frequent already cover half the cohort. We apply to them the
same logic that guided comorbidities and secondary diagnoses — preserve categories with real clinical
and sample weight, gathering the sparse tail into a single "other". This way the model learns from what
is common and relevant, without scattering across hundreds of nearly empty columns.

`AdmissionReasonName` (the "major area", already consolidated into ~60 categories) remains as is — it is
itself the by-system grouping of this same information, and enters whole.

In [ ]:
with open(CONJ_PATH, encoding='utf-8') as f:
    sets = json.load(f)
BENCHMARK = sets['benchmark']

TOP_N = 20
top_diag = df['AdmissionMainDiagnosisName'].value_counts().head(TOP_N).index
df['AdmissionMainDiagnosisName_grp'] = np.where(
    df['AdmissionMainDiagnosisName'].isin(top_diag),
    df['AdmissionMainDiagnosisName'], 'Outros_diagnostico_principal')

# replace the raw column with the grouped one in both variable lists
def usar_agrupada(lista):
    return [c if c != 'AdmissionMainDiagnosisName' else 'AdmissionMainDiagnosisName_grp' for c in lista]
CONJUNTO_1 = usar_agrupada(sets['set_1_escores'])
CONJUNTO_2 = usar_agrupada(sets['set_2_bruto'])

print(f'Main diagnosis: {df["AdmissionMainDiagnosisName"].nunique()} categories '
      f'-> {df["AdmissionMainDiagnosisName_grp"].nunique()} after grouping (top-{TOP_N} + other)')
print(f'Set 1: {len(CONJUNTO_1)} variables  |  Set 2: {len(CONJUNTO_2)} variables')

## 3 · The preparation mold: encoding, imputation, and tail handling

We gather into a single function the preparation of each set, so that train and test pass through exactly
the same mold — and, crucially, so that **nothing from the test leaks into training**. Three safeguards:

- **One-hot with a reference category** (`drop_first=True`): each text variable becomes 0/1 columns,
  one category serving as reference to avoid the redundancy that destabilizes logistic regression.
- **Median imputation**, learned *on training only* and applied to test — the missingness indicators
  created in Phase 2 preserve the memory of where data was absent.
- **Cap on the urea/creatinine ratio** (winsorization at the 99th percentile, computed on training only):
  its long and clinically real tail — very low creatinines in debilitated patients — would distort a
  linear model. The cap contains the extreme without discarding the patient. Next phase's gradient
  boosting will not need this, as it decides by splits, not by averages.

Scaling (`StandardScaler`) is also fitted on training only.

In [ ]:
CATEGORICAS = ['Gender','AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
               'AdmissionMainDiagnosisName_grp','Chronic Health Status Name','Status COVID 19',
               'OrigemPagamento']

def preparar_set(set, treino, teste, winsor_col='razao_ureia_creatinina', winsor_p=0.99):
    cats = [c for c in set if c in CATEGORICAS]
    nums = [c for c in set if c not in CATEGORICAS and c != 'Month Year']
    treino, teste = treino.copy(), teste.copy()

    # cap on the long tail — threshold learned on training only
    if winsor_col in nums:
        teto = treino[winsor_col].quantile(winsor_p)
        treino[winsor_col] = treino[winsor_col].clip(upper=teto)
        teste[winsor_col]  = teste[winsor_col].clip(upper=teto)

    # one-hot with a reference category
    Xtr = pd.get_dummies(treino[cats+nums], columns=cats, drop_first=True, dummy_na=False)
    Xte = pd.get_dummies(teste[cats+nums],  columns=cats, drop_first=True, dummy_na=False)
    # align columns: test inherits exactly the training columns
    Xtr, Xte = Xtr.align(Xte, join='left', axis=1, fill_value=0)

    # imputation + scaling, both fitted on training only
    imp = SimpleImputer(strategy='median')
    sc  = StandardScaler()
    Xtr_s = sc.fit_transform(imp.fit_transform(Xtr))
    Xte_s = sc.transform(imp.transform(Xte))
    return Xtr_s, Xte_s, Xtr.columns.tolist()

def treinar_avaliar(set, nome, treino, teste):
    Xtr, Xte, feat = preparar_set(set, treino, teste)
    ytr, yte = treino[ALVO].values, teste[ALVO].values
    clf = LogisticRegression(penalty='l2', C=1.0, class_weight=None,
                             max_iter=5000, solver='lbfgs')
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    return {'nome': nome, 'AUROC': roc_auc_score(yte, p), 'AUPRC': average_precision_score(yte, p),
            'Brier': brier_score_loss(yte, p), 'n_variaveis': len(feat)}, clf, p, yte

print('Preparation mold and modeling function defined.')

## 4 · Two validations

Two distinct questions about the trust the model deserves, each with its role:

**Temporal validation** is the primary verdict. It trains on the past (admissions up to 2022) and tests
on the future (2023 onward), honestly simulating real use — a model is always built on yesterday and
applied to tomorrow. It matters especially here, because the cohort itself changed over the decade
(mortality fell from ~28% in 2015 to ~10% in 2025, with the pandemic's mark visible along the way).
It is the validation TRIPOD-AI values and PROBAST demands when weighing risk of bias.

**Random validation by patient** is the stability reference. Since this is the unique-pair cohort —
one row per patient — splitting by row already respects patient independence. If the two validations
tell similar stories, that is reassuring; if they diverge, the difference is itself a portrait of the
data's temporal drift.

In [ ]:
# --- Temporal validation (primary) ---
treino_t = df[df['ano'] <  CORTE_TEMPORAL].copy()
teste_t  = df[df['ano'] >= CORTE_TEMPORAL].copy()
total_t = len(treino_t) + len(teste_t)
print(f'TEMPORAL VALIDATION — training (<{CORTE_TEMPORAL}): {len(treino_t)} ({len(treino_t)/total_t*100:.1f}%)  |  '
      f'test (>={CORTE_TEMPORAL}): {len(teste_t)} ({len(teste_t)/total_t*100:.1f}%)')
print(f'  training deaths: {treino_t[ALVO].mean()*100:.1f}%  |  test deaths: {teste_t[ALVO].mean()*100:.1f}%')

resultados_temporal = []
predicoes_temporal = {}
for conj, nome in [(CONJUNTO_1, 'Conjunto 1 (escores)'), (CONJUNTO_2, 'Conjunto 2 (brutos)')]:
    r, clf, p, yte = treinar_avaliar(conj, nome, treino_t, teste_t)
    resultados_temporal.append(r); predicoes_temporal[nome] = (p, yte)

# SAPS3 benchmark (probability already computed, simply read at test time)
saps3_p = (teste_t[BENCHMARK] / 100).values
yte = teste_t[ALVO].values
resultados_temporal.append({'nome': 'SAPS3 (benchmark)',
    'AUROC': roc_auc_score(yte, saps3_p), 'AUPRC': average_precision_score(yte, saps3_p),
    'Brier': brier_score_loss(yte, saps3_p), 'n_variaveis': '—'})
predicoes_temporal['SAPS3 (benchmark)'] = (saps3_p, yte)

tab_temporal = pd.DataFrame(resultados_temporal)
print()
print(tab_temporal.to_string(index=False))

In [ ]:
# ---- ROC and calibration curves — temporal validation ----
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
cores = {'Conjunto 1 (escores)': '#4C72B0', 'Conjunto 2 (brutos)': '#DD8452', 'SAPS3 (benchmark)': '#55A868'}

# ROC curves (discrimination)
ax = axes[0]
for nome, (p, y) in predicoes_temporal.items():
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, label=f'{nome} (AUROC={auc:.3f})', color=cores[nome], linewidth=2)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set_xlabel('1 - Especificidade (Falso Positivo)')
ax.set_ylabel('Sensibilidade (Verdadeiro Positivo)')
ax.set_title('ROC curves — temporal validation')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# Calibration curve (Brier, visualized)
ax = axes[1]
for nome, (p, y) in predicoes_temporal.items():
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy='quantile')
    brier = brier_score_loss(y, p)
    ax.plot(mean_pred, frac_pos, marker='o', label=f'{nome} (Brier={brier:.3f})', color=cores[nome], linewidth=2)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed death frequency')
ax.set_title('Calibration curve — temporal validation')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/fase3_roc_calibracao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {OUTDIR}/fase3_roc_calibracao_temporal.png")

In [ ]:
# --- Random validation by patient (reference) ---
# Index-admission cohort: one row per patient — a row-level train_test_split
# already equals a patient-level split, with no risk of between-group leakage.
treino_a, teste_a = train_test_split(df, test_size=0.25, random_state=42, stratify=df[ALVO])
total_a = len(treino_a) + len(teste_a)
print(f'RANDOM VALIDATION — training: {len(treino_a)} ({len(treino_a)/total_a*100:.1f}%)  |  '
      f'test: {len(teste_a)} ({len(teste_a)/total_a*100:.1f}%)  (outcome-stratified)')

resultados_aleatorio = []
predicoes_aleatorio = {}
for conj, nome in [(CONJUNTO_1, 'Conjunto 1 (escores)'), (CONJUNTO_2, 'Conjunto 2 (brutos)')]:
    r, clf, p, yte = treinar_avaliar(conj, nome, treino_a, teste_a)
    resultados_aleatorio.append(r); predicoes_aleatorio[nome] = (p, yte)

saps3_p = (teste_a[BENCHMARK] / 100).values
yte = teste_a[ALVO].values
resultados_aleatorio.append({'nome': 'SAPS3 (benchmark)',
    'AUROC': roc_auc_score(yte, saps3_p), 'AUPRC': average_precision_score(yte, saps3_p),
    'Brier': brier_score_loss(yte, saps3_p), 'n_variaveis': '—'})
predicoes_aleatorio['SAPS3 (benchmark)'] = (saps3_p, yte)

tab_aleatorio = pd.DataFrame(resultados_aleatorio)
print()
print(tab_aleatorio.to_string(index=False))

In [ ]:
# ---- ROC and calibration curves — random validation ----
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5.5))

# ROC curves (discrimination)
ax = axes2[0]
for nome, (p, y) in predicoes_aleatorio.items():
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, label=f'{nome} (AUROC={auc:.3f})', color=cores[nome], linewidth=2)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set_xlabel('1 - Especificidade (Falso Positivo)')
ax.set_ylabel('Sensibilidade (Verdadeiro Positivo)')
ax.set_title('ROC curves — random validation')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# Calibration curve (Brier, visualized)
ax = axes2[1]
for nome, (p, y) in predicoes_aleatorio.items():
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy='quantile')
    brier = brier_score_loss(y, p)
    ax.plot(mean_pred, frac_pos, marker='o', label=f'{nome} (Brier={brier:.3f})', color=cores[nome], linewidth=2)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed death frequency')
ax.set_title('Calibration curve — random validation')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/fase3_roc_calibracao_aleatoria.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {OUTDIR}/fase3_roc_calibracao_aleatoria.png")

## 5 · Reading the results — the first signal

These first results are baseline numbers, still without hyperparameter fine-tuning, without calibration,
without gradient boosting. Even so, a pattern already emerges clearly.

**Set 1** (ready-made scores) tracks SAPS3 closely — coherent, since it is made, in essence, of the same
scores SAPS3 summarizes. **Set 2** (raw components), in turn, distinguishes itself from both, with
superior discrimination and precision. If this pattern survives the next steps, it answers the project's
founding question: *yes, there is predictive signal beyond what SAPS3 already synthesizes* — the AI sees
something the ready-made formula does not reach.

The distance between the two validations also teaches. In the random split, Set 2 shines more (train and
test share the same era); in the more demanding temporal split, the shine dims a little — the honest
price of predicting the future from the past, in a cohort that changed over ten years. This difference is
not a flaw: it is the frank measure of how much time weighs on the model, and it is exactly what rigorous
evaluation should reveal rather than hide.

What we have **not** yet examined, and the next phase will demand: **calibration** (a model can rank risk
well and still get the probability wrong — dangerous at the bedside), the **DeLong test** to say whether
the advantage over SAPS3 is statistically firm, and **gradient boosting with SHAP** to seek more
performance without losing clinical legibility.

## 6 · Store the baseline results
The two performance tables, preserved for direct comparison in the next phase.

In [ ]:
tab_temporal.insert(0, 'validacao', 'temporal')
tab_aleatorio.insert(0, 'validacao', 'aleatoria_por_paciente')
resultados = pd.concat([tab_temporal, tab_aleatorio], ignore_index=True)

res_path = f'{OUTDIR}/modelo_A_fase3_linha_de_base.csv'
resultados.to_csv(res_path, index=False, encoding='utf-8-sig')

fig_temporal_path = f'{OUTDIR}/fase3_roc_calibracao_temporal.png'
fig_aleatoria_path = f'{OUTDIR}/fase3_roc_calibracao_aleatoria.png'
fig.savefig(fig_temporal_path, dpi=150, bbox_inches='tight')
fig2.savefig(fig_aleatoria_path, dpi=150, bbox_inches='tight')

print(f'Resultados salvos: {res_path}')
print(f'Figure (temporal validation) saved: {fig_temporal_path}')
print(f'Figure (random validation) saved: {fig_aleatoria_path}')
print()
print(resultados.to_string(index=False))

---
### What this phase established
- The base enters sanitized of Excel artifacts.
- The main diagnosis was grouped with the same logic as previous phases.
- Preparation (one-hot, imputation, tail cap, scaling) respects the train/test boundary, without
  leakage — each piece fitted only where the model is allowed to look.
- The baseline — regularized logistic regression — was measured on both sets, under two validations,
  against the SAPS3 yardstick.

### Next step (Phase 4)
- **Gradient boosting** (XGBoost / LightGBM / CatBoost) on both sets, with hyperparameter tuning by
  stratified cross-validation — on training only.
- **SHAP** on the best model, returning clinical legibility to boosting: which variables push risk, and
  in which patients.
- **DeLong test** against SAPS3, to state with statistical grounding whether the advantage holds.
- **Calibration** (decomposed Brier score, calibration curve, intercept and slope) and **decision-curve
  analysis**, to weigh not only accuracy but net clinical benefit.

### Limitation of this phase — transfer bias
The baseline results (AUROC, Brier, AUPRC) were computed on the **primary definition** of
`mortalidade_hospitalar`. Cardiovascular patients transferred before death are coded as survivors under
this definition, which underestimates true mortality in that subgroup and may slightly inflate the
model's discrimination metrics ("difficult" cases coded as survivors agree with the coded outcome, but
not with the real one).

When the parallel analysis with `mortalidade_hospitalar_ajustada` is run in Phase 5, this phase's results
should be compared to quantify the bias's impact. Record in PROBAST, outcome domain, that this phase's
numbers are subject to revision after the adjustment.